# 🌷 LILY VIDEO STUDIO V3 — PREWARM BUILD

Fresh Kaggle build for iPhone. **Enable a Kaggle GPU before running Cell 1.**

Run the four cells from top to bottom. Cell 3 is intentionally the long one: it downloads and verifies all three video models **before** the Lily UI opens.

Models: **LTX-2.3 Distilled 1.1**, **Wan 2.2 I2V**, and **HunyuanVideo 1.5 I2V**.

Model files live in Kaggle temporary session storage, so a completely new Kaggle session will need to download them again.


## 1 — Build the playground

Checks the NVIDIA GPU, creates the cache folders, clones WanGP, and installs system video libraries.


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('NO NVIDIA GPU. In Kaggle open Settings → Accelerator → GPU, then restart the session and run this cell again.')

subprocess.run(['nvidia-smi'], check=True)

ROOT = Path('/kaggle/working/Wan2GP')
DATA = Path('/kaggle/temp/Wan2GP-data')
CKPTS = DATA / 'ckpts'
LORAS = DATA / 'loras'
CACHE = DATA / 'cache'
OUTPUTS = Path('/kaggle/working/Wan2GP-outputs')

for p in (DATA, CKPTS, LORAS, CACHE, OUTPUTS):
    p.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(CACHE / 'huggingface')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE / 'huggingface' / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(CACHE / 'huggingface' / 'transformers')
os.environ['TORCH_HOME'] = str(CACHE / 'torch')
os.environ['XDG_CACHE_HOME'] = str(CACHE / '.cache')
os.environ['WAN_CACHE_DIR'] = str(CACHE)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

REPO = 'https://github.com/deepbeepmeep/Wan2GP.git'
if not (ROOT / '.git').exists():
    if ROOT.exists():
        shutil.rmtree(ROOT)
    subprocess.run(['git', 'clone', '--depth', '1', REPO, str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=False)

def attach(repo_dir: Path, storage_dir: Path):
    storage_dir.mkdir(parents=True, exist_ok=True)
    if repo_dir.is_symlink():
        if repo_dir.resolve() == storage_dir.resolve():
            return
        repo_dir.unlink()
    elif repo_dir.exists():
        for item in list(repo_dir.iterdir()):
            dest = storage_dir / item.name
            if not dest.exists():
                shutil.move(str(item), str(dest))
        shutil.rmtree(repo_dir)
    repo_dir.symlink_to(storage_dir, target_is_directory=True)

attach(ROOT / 'ckpts', CKPTS)
attach(ROOT / 'loras', LORAS)
attach(ROOT / 'outputs', OUTPUTS)

env = os.environ.copy()
env['DEBIAN_FRONTEND'] = 'noninteractive'
prefix = [] if os.geteuid() == 0 else ['sudo']
subprocess.run(prefix + ['apt-get', 'update', '-qq'], check=True, env=env)
subprocess.run(
    prefix + ['apt-get', 'install', '-y', '--no-install-recommends',
              'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'],
    check=True, env=env,
)
print('✅ Cell 1 complete: GPU + WanGP + system libraries ready.')


## 2 — Install WanGP Python dependencies

This is the **insanely long dependency cell**. Let it finish completely.

It probes Kaggle's existing CUDA Torch in a separate Python process, pins those versions, installs WanGP, then validates NumPy/Torch/WanGP dependencies in another fresh process. This avoids the NumPy hot-swap problem from the earlier notebook.


In [ ]:
import json, os, subprocess, sys
from pathlib import Path

probe_code = "\n".join([
    "import json",
    "out = {}",
    "for name in ('torch', 'torchvision', 'torchaudio'):",
    "    try:",
    "        module = __import__(name)",
    "        out[name] = getattr(module, '__version__', None)",
    "    except Exception:",
    "        out[name] = None",
    "print(json.dumps(out))",
])
probe_raw = subprocess.check_output([sys.executable, '-c', probe_code], text=True)
installed = json.loads(probe_raw.strip().splitlines()[-1])
if not installed.get('torch'):
    raise RuntimeError('Kaggle Torch could not be detected. Restart the GPU session before continuing.')

constraints = CACHE / 'kaggle-torch-constraints.txt'
pins = [f"torch=={installed['torch'].split('+', 1)[0]}"]
for name in ('torchvision', 'torchaudio'):
    version = installed.get(name)
    if version:
        pins.append(f"{name}=={version.split('+', 1)[0]}")
constraints.write_text('\n'.join(pins) + '\n')
print('Protecting Kaggle CUDA packages with:')
print(constraints.read_text())

env = os.environ.copy()
env['PIP_NO_CACHE_DIR'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', 'setuptools', 'wheel'], check=True, env=env)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
    '--upgrade-strategy', 'only-if-needed',
    '-r', str(ROOT / 'requirements.txt'), '-c', str(constraints)
], check=True, env=env)

target = ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
if target.exists():
    text = target.read_text()
    patched = text.replace("matplotlib.use('TkAgg')", "matplotlib.use('Agg')")
    if patched != text:
        target.write_text(patched)

validate_code = "\n".join([
    "import numpy, torch, mmgp, rembg, gradio",
    "assert torch.cuda.is_available(), 'Torch installed but CUDA is unavailable.'",
    "print('NumPy:', numpy.__version__)",
    "print('Torch:', torch.__version__)",
    "print('CUDA:', torch.version.cuda)",
    "print('GPU:', torch.cuda.get_device_name(0))",
    "print('mmgp/rembg/gradio imports: OK')",
])
subprocess.run([sys.executable, '-c', validate_code], check=True, env=os.environ.copy())
print('✅ Cell 2 complete: WanGP Python stack installed and validated.')


## 3 — 📦 Download + verify ALL 3 models

This is the **model download stage**. It intentionally runs one tiny 2-second warm-up through each model. That makes WanGP fetch the exact checkpoint/preload files it needs and verifies the model actually runs.

Do not stop this cell just because it is taking a while. The first fresh session can be very slow here.


In [ ]:
import os, subprocess, sys, urllib.request

PREWARM = ROOT / 'lily_prewarm_models.py'
PREWARM_URL = 'https://raw.githubusercontent.com/benruiz1024-ops/hi/main/lily_prewarm_models.py'
urllib.request.urlretrieve(PREWARM_URL, PREWARM)
print('📦 Prewarm script updated from GitHub.')
print('Downloading/verifying LTX-2.3, Wan 2.2 I2V, and HunyuanVideo 1.5 I2V…')
print('This is supposed to be the long stage.\n')
subprocess.run([sys.executable, '-u', str(PREWARM)], cwd=str(ROOT), env=os.environ.copy(), check=True)
print('\n✅ Cell 3 complete: all three models are ready for this Kaggle session.')


## 4 — 🌷 Launch Lily Video Studio

Keep this cell running while you use the site. When the public **gradio.live** link appears, tap it on your iPhone.


In [ ]:
import os, subprocess, sys, urllib.request

STUDIO = ROOT / 'lily_video_studio.py'
STUDIO_URL = 'https://raw.githubusercontent.com/benruiz1024-ops/hi/main/lily_video_studio.py'
urllib.request.urlretrieve(STUDIO_URL, STUDIO)
print('🌷 Lily Video Studio updated from GitHub.')
print('Launching… wait for the public gradio.live link. Keep this cell running.')
subprocess.run([sys.executable, '-u', str(STUDIO)], cwd=str(ROOT), env=os.environ.copy(), check=False)
